In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

In [ ]:
file_515 = r"J:\ctgroup\Edward\DATA\VMI\20250408\calibrated\po_500mW_ell_calibrated.h5"
file_1030 = r"J:\ctgroup\Edward\DATA\VMI\20250307\Intensity Scan\4_calibrated.h5"

df_515 = pd.read_hdf(file_515)
df_1030 = pd.read_hdf(file_1030)
pmax = 0.6
df_515 = df_515[(df_515['px'] ** 2 + df_515['py'] ** 2 + df_515['pz'] ** 2) < pmax ** 2]
df_1030 = df_1030[(df_1030['px'] ** 2 + df_1030['py'] ** 2 + df_1030['pz'] ** 2) < pmax ** 2]

mq_range = (0, 70)
df_515 = df_515[(df_515['m/q'] > mq_range[0]) & (df_515['m/q'] < mq_range[1])]
df_1030 = df_1030[(df_1030['m/q'] > mq_range[0]) & (df_1030['m/q'] < mq_range[1])]

df_515['pr'] = np.sqrt(df_515['px'] ** 2 + df_515['py'] ** 2 + df_515['pz'] ** 2)
df_1030['pr'] = np.sqrt(df_1030['px'] ** 2 + df_1030['py'] ** 2 + df_1030['pz'] ** 2)

In [ ]:
gates = {
    "C2H4+": (20, 33),
    "C2H3O+": (36, 47),
    "Parent PO+": (50, 60),
}

px.histogram(df_515, x='m/q', nbins=500, title='515nm Mass Spectrum', labels={'m/q': 'm/q (a.u.)'}).show()
px.histogram(df_1030, x='m/q', nbins=500, title='1030nm Mass Spectrum', labels={'m/q': 'm/q (a.u.)'}).show()

In [ ]:
for name, (low, high) in gates.items():
    df_515.loc[(df_515['m/q'] > low) & (df_515['m/q'] < high), 'ion'] = name
    df_1030.loc[(df_1030['m/q'] > low) & (df_1030['m/q'] < high), 'ion'] = name

r_grid = np.linspace(0.001, 0.5, 50)

hists_515 = [
    np.histogram(
            df_515[df_515['ion'] == i]['pr'],
            weights=1 / df_515[df_515['ion'] == i]['pr'] ** 2,
            bins=50, range=(0, 0.5), density=True
    )[0]
    for i in gates.keys()
]
hists_1030 = [
    np.histogram(
            df_1030[df_1030['ion'] == i]['pr'],
            weights=1 / df_1030[df_1030['ion'] == i]['pr'] ** 2,
            bins=50, range=(0, 0.5), density=True
    )[0]
    for i in gates.keys()
]

fig = make_subplots(2, 1, subplot_titles=("515nm", "1030nm"), shared_xaxes=True, x_title="pr (a.u.)",
                    y_title="Signal (arb. units)")
[fig.add_scatter(
        x=r_grid,
        y=hist_515,
        line_color=color,
        name=f"{name}",
        row=1, col=1,
) for hist_515, name, color in zip(hists_515, gates.keys(), ['red', 'green', 'blue'])]
[fig.add_scatter(
        x=r_grid,
        y=hist_1030,
        line_color=color,
        row=2, col=1,
        showlegend=False,
) for hist_1030, name, color in zip(hists_1030, gates.keys(), ['red', 'green', 'blue'])]
fig.update_layout(
        width=800,
        height=800,
        template='simple_white+presentation',
        legend_orientation="h",
        legend_x=0,
        legend_y=1.1,
)
fig.show()


In [ ]:
import numpy as np
from plotly.subplots import make_subplots

fig2 = make_subplots(rows=2, cols=len(gates), subplot_titles=list(gates.keys()),
                     x_title="px", y_title=" (<--1030) py (515-->)\n", horizontal_spacing=0.1, vertical_spacing=0.1)
pz_gate = (-0.1, 0.1)

bins = 256
x_range = [-pmax, pmax]
y_range = [-pmax, pmax]
zmax = 40

for col, (name, (low, high)) in enumerate(gates.items(), start=1):
    df_gate_515 = df_515[
        (df_515['m/q'] > low) & (df_515['m/q'] < high) & (df_515['pz'] > pz_gate[0]) & (df_515['pz'] < pz_gate[1])]
    df_gate_1030 = df_1030[
        (df_1030['m/q'] > low) & (df_1030['m/q'] < high) & (df_1030['pz'] > pz_gate[0]) & (df_1030['pz'] < pz_gate[1])]

    # compute 2D histograms with numpy
    H515, xedges, yedges = np.histogram2d(df_gate_515['px'], df_gate_515['py'], bins=bins, range=[x_range, y_range])
    H1030, _, _ = np.histogram2d(df_gate_1030['px'], df_gate_1030['py'], bins=bins, range=[x_range, y_range])

    # bin centers
    xcenters = 0.5 * (xedges[:-1] + xedges[1:])
    ycenters = 0.5 * (yedges[:-1] + yedges[1:])

    # add heatmap traces (transpose H so x->columns, y->rows)
    show_colorbar = (col == len(gates))  # show single colorbar on the right-most column
    fig2.add_trace(
            px.imshow(
                    np.nan_to_num(np.log10(H515).T, neginf=0),
                    x=xcenters,
                    y=ycenters,
                    color_continuous_scale='Inferno',
                    labels={'color': 'Counts'},
                    title=f"1030nm {name}",
            ).data[0],
            row=1, col=col,
    )
    fig2.add_trace(
            px.imshow(
                    np.nan_to_num(np.log10(H1030).T, neginf=0),
                    x=xcenters,
                    y=ycenters,
                    color_continuous_scale='Inferno',
                    labels={'color': 'Counts'},
                    title=f"515nm {name}",
            ).data[0],
            row=2, col=col,
    )

# make individual subplots square by anchoring each y-axis to its x-axis
ncols = len(gates)
# for row in (1, 2):
#     for col in range(1, ncols + 1):
#         # update_yaxes accepts row/col, and we anchor to the corresponding x axis
#         fig2.update_yaxes(scaleanchor=f'x{(row - 1) * ncols + col}', scaleratio=1, row=row, col=col)


fig2.update_layout(
        width=350 * ncols,  # width scales with number of columns so panels remain roughly square
        height=600,  # two rows => each cell will be approximately square given the scaleanchor above
        template='simple_white+presentation',
        title_text="Momentum Distributions (log scale)",
        showlegend=False,
        margin=dict(t=100, b=50, l=100, r=100),
        coloraxis_colorscale=px.colors.sequential.Inferno,
)

fig2.show()


In [ ]:
import numpy as np
from plotly.subplots import make_subplots

fig2 = make_subplots(rows=2, cols=len(gates), subplot_titles=list(gates.keys()),
                     x_title="px", y_title=" (<--1030) py (515-->)\n", horizontal_spacing=0.1, vertical_spacing=0.1)
pz_gate = (-0.1, 0.1)

bins = 256
x_range = [-pmax, pmax]
y_range = [-pmax, pmax]
zmax = 40

for col, (name, (low, high)) in enumerate(gates.items(), start=1):
    df_gate_515 = df_515[
        (df_515['m/q'] > low) & (df_515['m/q'] < high) & (df_515['pz'] > pz_gate[0]) & (df_515['pz'] < pz_gate[1])]
    df_gate_1030 = df_1030[
        (df_1030['m/q'] > low) & (df_1030['m/q'] < high) & (df_1030['pz'] > pz_gate[0]) & (df_1030['pz'] < pz_gate[1])]

    # compute 2D histograms with numpy
    H515, xedges, yedges = np.histogram2d(df_gate_515['px'], df_gate_515['py'], bins=bins, range=[x_range, y_range])
    H1030, _, _ = np.histogram2d(df_gate_1030['px'], df_gate_1030['py'], bins=bins, range=[x_range, y_range])

    # bin centers
    xcenters = 0.5 * (xedges[:-1] + xedges[1:])
    ycenters = 0.5 * (yedges[:-1] + yedges[1:])

    # add heatmap traces (transpose H so x->columns, y->rows)
    show_colorbar = (col == len(gates))  # show single colorbar on the right-most column
    fig2.add_trace(
            px.imshow(
                    H515.T / np.max(H515),
                    x=xcenters,
                    y=ycenters,
                    color_continuous_scale='Inferno',
                    labels={'color': 'Counts'},
                    title=f"1030nm {name}",
            ).data[0],
            row=1, col=col,
    )
    fig2.add_trace(
            px.imshow(
                    H1030.T / np.max(H1030),
                    x=xcenters,
                    y=ycenters,
                    color_continuous_scale='Inferno',
                    labels={'color': 'Counts'},
                    title=f"515nm {name}",
            ).data[0],
            row=2, col=col,
    )

# make individual subplots square by anchoring each y-axis to its x-axis
ncols = len(gates)
# for row in (1, 2):
#     for col in range(1, ncols + 1):
#         # update_yaxes accepts row/col, and we anchor to the corresponding x axis
#         fig2.update_yaxes(scaleanchor=f'x{(row - 1) * ncols + col}', scaleratio=1, row=row, col=col)


fig2.update_layout(
        width=350 * ncols,  # width scales with number of columns so panels remain roughly square
        height=600,  # two rows => each cell will be approximately square given the scaleanchor above
        template='simple_white+presentation',
        title_text="Momentum Distributions (linear scale)",
        showlegend=False,
        margin=dict(t=100, b=50, l=100, r=100),
        coloraxis_colorscale=px.colors.sequential.Inferno,
)

fig2.show()


In [ ]:
for name, (low, high) in gates.items():
    df_515.loc[(df_515['m/q'] > low) & (df_515['m/q'] < high), 'ion'] = name
    df_1030.loc[(df_1030['m/q'] > low) & (df_1030['m/q'] < high), 'ion'] = name

df_515['is_zero_energy'] = df_515['px'] ** 2 + df_515['py'] ** 2 < 0.04 ** 2
df_1030['is_zero_energy'] = df_1030['px'] ** 2 + df_1030['py'] ** 2 < 0.04 ** 2

df_515['is_reasonable'] = df_515['pr'] < 0.6
df_1030['is_reasonable'] = df_1030['pr'] < 0.6

df_515 = df_515[df_515['is_reasonable']]
df_1030 = df_1030[df_1030['is_reasonable']]

df_515_summary = df_515.groupby('ion')['is_zero_energy'].mean().reset_index()
df_1030_summary = df_1030.groupby('ion')['is_zero_energy'].mean().reset_index()
summary = pd.merge(df_515_summary, df_1030_summary, on='ion', suffixes=('_515nm', '_1030nm'))
summary

In [ ]:
for name, (low, high) in gates.items():
    df_515.loc[(df_515['m/q'] > low) & (df_515['m/q'] < high), 'ion'] = name
    df_1030.loc[(df_1030['m/q'] > low) & (df_1030['m/q'] < high), 'ion'] = name

df_515['energy'] = df_515['pr'] ** 2 / 2
df_1030['energy'] = df_1030['pr'] ** 2 / 2
e_grid = np.linspace(0.001, .15, 200)
hists_515 = [np.histogram(df_515[df_515['ion'] == i]['energy'], bins=200, range=(0, .15), density=True)[0] for i in
             gates.keys()]
hists_1030 = [np.histogram(df_1030[df_1030['ion'] == i]['energy'], bins=200, range=(0, .15), density=True)[0] for i in
              gates.keys()]
fig = make_subplots(2, 1, subplot_titles=("515nm", "1030nm"), shared_xaxes=True, x_title="Energy (a.u.)",
                    y_title="Signal (arb. units, density normalized)")
[fig.add_scatter(
        x=e_grid,
        y=hist_515,
        line_color=color,
        name=f"{name}",
        row=1, col=1,
) for hist_515, name, color in zip(hists_515, gates.keys(), ['red', 'green', 'blue'])]
[fig.add_scatter(
        x=e_grid,
        y=hist_1030,
        line_color=color,
        row=2, col=1,
        showlegend=False,
) for hist_1030, name, color in zip(hists_1030, gates.keys(), ['red', 'green', 'blue'])]
fig.update_layout(
        width=800,
        height=800,
        template='simple_white+presentation',
        legend_orientation="h",
        legend_x=0,
        legend_y=1.1,
)
fig.show()


In [ ]:
h2d_515 = np.histogram2d(
        df_515['m/q'], df_515['energy'],
        bins=[500, 200],
        range=[[0, 70], [0, 0.15]],
)[0]

hmq_515 = np.histogram(df_515['m/q'], bins=500, range=(0, 70))[0]
hpr_515 = np.histogram(df_515['energy'], bins=200, range=(0, 0.15))[0]

px.imshow(
        h2d_515.T,
        origin='lower',
).show()
px.imshow(
        np.outer(hmq_515, hpr_515).T,
        origin='lower',
).show()

cov = h2d_515 - np.outer(hmq_515, hpr_515) / (len(df_515))
px.imshow(
        cov.T,
        origin='lower',
        color_continuous_scale='RdBu',
        color_continuous_midpoint=0,
).show()

h2d_1030 = np.histogram2d(
        df_1030['m/q'], df_1030['energy'],
        bins=[500, 200],
        range=[[0, 70], [0, 0.15]],
)[0]

hmq_1030 = np.histogram(
        df_1030['m/q'], bins=500, range=(0, 70)
)[0]

hpr_1030 = np.histogram(
        df_1030['energy'], bins=200, range=(0, 0.15)
)[0]

px.imshow(
        h2d_1030.T,
        origin='lower',
).show()
px.imshow(
        np.outer(hmq_1030, hpr_1030).T,
        origin='lower',
).show()
cov = h2d_1030 - np.outer(hmq_1030, hpr_1030) / (len(df_1030))
px.imshow(
        cov.T,
        origin='lower',
        color_continuous_scale='RdBu',
        color_continuous_midpoint=0,
).show()
